In [1]:
from google.colab import files
uploaded = files.upload()


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
resume = pd.read_excel("/content/labeled_resumes.xlsx")

In [ ]:
resume.columns = resume.columns.str.strip()
resume = resume.loc[:, ~resume.columns.str.startswith('Unnamed')]


In [ ]:
resume

,NAME,AGE,LOCATION,EMAILID,EDU,YOE,TECH KN.,HANDS ON PROJECT,CERTIFICATIONS,combined_text,MATCHED
0,RIYA,25,PUNE,riya20@gmail.com,pg,3,data science,myntra analysis,niit,pg data science myntra analysis niit,1
1,Amanprret Kaur,26,Mohali,aman@gmail.com,ug,4,data science,e-commerce website,infosys springboard,ug data science e-commerce website infosys spr...,0
2,Daman,23,Chandigarh,Daman23@gmail.com,ug,1,data science,myntra analysis,niit,ug data science myntra analysis niit,0
3,Surbhi,24,Mohali,surbhi@gmail.com,pg,2,data science,netflix recommendeation,amazon,pg data science netflix recommendeation amazon,1
4,Sourav,25,Mohali,Sourav@gmail.com,ug,3,data science,fake news detection,google,ug data science fake news detection google,0
5,Yakshi,25,Noida,yakshi23@gmail.com,pg,3,data science,house price prediction,nit,pg data science house price prediction nit,1
6,Shubham,25,Banguluru,Shubham_14@gmail.com,ug,4,data science,stock market price analysis,google,ug data science stock market price analysis go...,0
7,Dr. Sita,29,Gurugaram,Sita@gmail.com,p.hd,5,data science,loan prediction analysis,infosys springboard,p.hd data science loan prediction analysis inf...,1
8,Bavneet,25,Gurugaram,Bavneet@gmail.com,pg,3,data science,ai resume maker,infosys springboard,pg data science ai resume maker infosys spring...,1
9,Harshdeep Singh,26,Pune,Harsheet@gmail.com,pg,4,data science,mental health analysis,niit,pg data science mental health analysis niit,1


In [ ]:
# Remove rows where ALL columns are NaN
resume = resume.dropna(how='all')

# Optionally, remove rows where key columns (like combined_text) are NaN
resume = resume.dropna(subset=['EDU', 'TECH KN.', 'HANDS ON PROJECT', 'CERTIFICATIONS'], how='all')


In [ ]:
resume

,NAME,AGE,LOCATION,EMAILID,EDU,YOE,TECH KN.,HANDS ON PROJECT,CERTIFICATIONS,combined_text,MATCHED
0,RIYA,25,PUNE,riya20@gmail.com,pg,3,data science,myntra analysis,niit,pg data science myntra analysis niit,1
1,Amanprret Kaur,26,Mohali,aman@gmail.com,ug,4,data science,e-commerce website,infosys springboard,ug data science e-commerce website infosys spr...,0
2,Daman,23,Chandigarh,Daman23@gmail.com,ug,1,data science,myntra analysis,niit,ug data science myntra analysis niit,0
3,Surbhi,24,Mohali,surbhi@gmail.com,pg,2,data science,netflix recommendeation,amazon,pg data science netflix recommendeation amazon,1
4,Sourav,25,Mohali,Sourav@gmail.com,ug,3,data science,fake news detection,google,ug data science fake news detection google,0
5,Yakshi,25,Noida,yakshi23@gmail.com,pg,3,data science,house price prediction,nit,pg data science house price prediction nit,1
6,Shubham,25,Banguluru,Shubham_14@gmail.com,ug,4,data science,stock market price analysis,google,ug data science stock market price analysis go...,0
7,Dr. Sita,29,Gurugaram,Sita@gmail.com,p.hd,5,data science,loan prediction analysis,infosys springboard,p.hd data science loan prediction analysis inf...,1
8,Bavneet,25,Gurugaram,Bavneet@gmail.com,pg,3,data science,ai resume maker,infosys springboard,pg data science ai resume maker infosys spring...,1
9,Harshdeep Singh,26,Pune,Harsheet@gmail.com,pg,4,data science,mental health analysis,niit,pg data science mental health analysis niit,1


In [ ]:
for col in ['EDU', 'TECH KN.', 'HANDS ON PROJECT', 'CERTIFICATIONS']:
    resume[col] = resume[col].astype(str).str.lower().str.strip()


In [ ]:
resume['combined_text'] = (
    resume['EDU'] + ' ' +
    resume['TECH KN.'] + ' ' +
    resume['HANDS ON PROJECT'] + ' ' +
    resume['CERTIFICATIONS'])

In [ ]:
if 'MATCHED' not in resume.columns:
    resume['MATCHED'] = None

In [ ]:
job_desc = """We are looking for a Data Engineering Developer with 3+ years of experience
building data-driven backend systems using Python, Django, and REST APIs.
Experience deploying data pipelines or analytics services in production,
using Docker and cloud environments (AWS/GCP), is required."""

In [ ]:
!ls


labeled_resumes.xlsx  sample_data


In [ ]:
# Drop rows where MATCHED is missing
resume = resume.dropna(subset=['MATCHED'])



In [ ]:

# Convert MATCHED to integer
resume['MATCHED'] = resume['MATCHED'].astype(int)

In [ ]:

# Optional: verify
print(resume['MATCHED'].value_counts())
print(resume['MATCHED'].dtype)

MATCHED
0    24
1     6
Name: count, dtype: int64
int64


In [ ]:
# Example flow so you can follow it properly
X = resume['combined_text']
y = resume['MATCHED']

# Split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Vectorize
vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)


In [ ]:
print("Train class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())


Train class distribution:
 MATCHED
0    19
1     5
Name: count, dtype: int64
Test class distribution:
 MATCHED
0    5
1    1
Name: count, dtype: int64


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "RandomForest": RandomForestClassifier(class_weight='balanced'),
    "SVM": SVC(class_weight='balanced')
}

In [ ]:
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    print(f"\n🔹🔹 {name} Results 🔹🔹")
    print("✅ Accuracy:", accuracy_score(y_test, y_pred))
    print("📊 Classification Report:\n", classification_report(y_test, y_pred))
    print("🟩 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



🔹🔹 LogisticRegression Results 🔹🔹
✅ Accuracy: 1.0
📊 Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         1

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

🟩 Confusion Matrix:
 [[5 0]
 [0 1]]

🔹🔹 RandomForest Results 🔹🔹
✅ Accuracy: 1.0
📊 Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         1

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

🟩 Confusion Matrix:
 [[5 0]
 [0 1]]

🔹🔹 SVM Results 🔹🔹
✅ Accuracy: 1.0
📊 Classification Report:
               precision    recall  f1-score   support

           0

In [ ]:
import joblib

# Save model
joblib.dump(model, "resume_ranker_model.pkl")

# Save vectorizer
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("✅ Model and vectorizer saved!")


✅ Model and vectorizer saved!


In [ ]:
import pandas as pd
import joblib


In [ ]:
# Load your saved model and vectorizer
model = joblib.load("resume_ranker_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")


In [ ]:
from google.colab import files

files.download("resume_ranker_model.pkl")
files.download("tfidf_vectorizer.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files


files.download("tfidf_vectorizer.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
resume_df = pd.read_excel("labeled_resumes.xlsx")
print("✅ Resume data loaded!")
print(resume_df.head())


✅ Resume data loaded!
             NAME  AGE    LOCATION            EMAILID EDU  YOE      TECH KN.  \
0            RIYA   25        PUNE   riya20@gmail.com  pg    3  data science   
1  Amanprret Kaur   26      Mohali     aman@gmail.com  ug    4  data science   
2           Daman   23  Chandigarh  Daman23@gmail.com  ug    1  data science   
3          Surbhi   24      Mohali   surbhi@gmail.com  pg    2  data science   
4          Sourav   25      Mohali   Sourav@gmail.com  ug    3  data science   

          HANDS ON PROJECT       CERTIFICATIONS  \
0          myntra analysis                 niit   
1       e-commerce website  infosys springboard   
2          myntra analysis                 niit   
3  netflix recommendeation               amazon   
4      fake news detection               google   

                                       combined_text  MATCHED  
0               pg data science myntra analysis niit        1  
1  ug data science e-commerce website infosys spr...        0 

In [ ]:
def compute_rank_score(row):
    score = 0

    # 🎓 Education weight (improved)
    edu = str(row['EDU']).lower()
    if edu in ['p.hd', 'phd']:
        score += 40
    elif edu == 'pg':
        score += 30
    elif edu == 'ug':
        score += 10

    # ⏳ YOE
    yoe = row['YOE']
    if yoe >= 3:
        score += 30
    elif yoe == 2:
        score += 20
    else:
        score += 10

    # 🛠 TECH
    tech = str(row['TECH KN.']).lower()
    if 'data science' in tech:
        score += 30
    elif 'software development' in tech:
        score += 20
    elif 'web development' in tech:
        score += 15

    # 🌟 Project & cert bonuses
    proj = str(row['HANDS ON PROJECT']).lower()
    cert = str(row['CERTIFICATIONS']).lower()

    if 'ai' in proj:
        score += 10
    if 'infosys' in cert or 'niit' in cert:
        score += 5

    return score


In [ ]:
resume_df['rank_score'] = resume_df.apply(compute_rank_score, axis=1)
ranked_resumes = resume_df.sort_values(by='rank_score', ascending=False)
print(ranked_resumes[['NAME', 'TECH KN.', 'HANDS ON PROJECT', 'rank_score']].head())


              NAME      TECH KN.          HANDS ON PROJECT  rank_score
8          Bavneet  data science           ai resume maker         105
7         Dr. Sita  data science  loan prediction analysis         105
0             RIYA  data science           myntra analysis          95
9  Harshdeep Singh  data science    mental health analysis          95
5           Yakshi  data science    house price prediction          90


In [ ]:
print(ranked_resumes[['NAME', 'EDU', 'YOE', 'TECH KN.', 'HANDS ON PROJECT', 'rank_score']].head(10))


               NAME   EDU  YOE              TECH KN.  \
8           Bavneet    pg    3          data science   
7          Dr. Sita  p.hd    5          data science   
0              RIYA    pg    3          data science   
9   Harshdeep Singh    pg    4          data science   
5            Yakshi    pg    3          data science   
27          rishabh    pg    3       web development   
23          Divish    phd    4       web development   
19          Pranjal    pg    3  software development   
12  Ramandeep Singh    pg    3  software development   
18          Ridhima    pg    3  software development   

                   HANDS ON PROJECT  rank_score  
8                   ai resume maker         105  
7          loan prediction analysis         105  
0                   myntra analysis          95  
9            mental health analysis          95  
5            house price prediction          90  
27  ai blog title generator web app          90  
23                  recipe book a

In [ ]:
test_df = pd.DataFrame([
    {
        "NAME": "Test PhD Senior",
        "EDU": "PhD",
        "YOE": 5,
        "TECH KN.": "data science",
        "HANDS ON PROJECT": "AI fraud detection",
        "CERTIFICATIONS": "Infosys"
    },
    {
        "NAME": "Test UG Junior",
        "EDU": "UG",
        "YOE": 1,
        "TECH KN.": "web development",
        "HANDS ON PROJECT": "blog website",
        "CERTIFICATIONS": "None"
    }
])

# Apply your scoring function
test_df['rank_score'] = test_df.apply(compute_rank_score, axis=1)
print(test_df[['NAME', 'rank_score']])


              NAME  rank_score
0  Test PhD Senior         115
1   Test UG Junior          35
